
# Property Dataset Expansion Notebook  
## Sales and rentals expansion to 10,000 rows each

In this notebook, I expand my two cleaned South African property datasets:

- `clean_sales_data.csv`
- `clean_rentals_data.csv`

My goal is to move from roughly **1,000 real rows** per dataset to **10,000 rows** per dataset **without changing the original column structure**.

## Why I am doing this

For a project of this size, I want enough data to support:

- stronger feature engineering
- more stable model training
- suburb and property-type variation
- time-based train / validation / test splitting
- cross-validation inside the training set

But I also need to stay realistic.

### Important modeling logic

I do **not** want to pretend synthetic rows are the same as freshly scraped market data.  
So I use this dataset expansion for:

- training augmentation
- stress-testing the modeling pipeline
- early experimentation

I still keep:

- **validation data real-only where possible**
- **test data untouched for final evaluation**

That means this notebook helps me create larger, model-ready training data while preserving a proper evaluation strategy.


In [17]:

import pandas as pd
import numpy as np
import re
from pathlib import Path

np.random.seed(42)
pd.set_option("display.max_columns", 100)



## 1. Load the exact two cleaned CSV files

I load the two attached datasets exactly as they are.  
The expansion process must preserve:

- the exact column names
- the exact column order
- the overall business meaning of the fields


In [19]:

sales_path = Path("data\processed\clean_sales_data.csv")
rentals_path = Path("data\processed\clean_rentals_data.csv")

df_sales = pd.read_csv(sales_path)
df_rentals = pd.read_csv(rentals_path)

print("Sales shape   :", df_sales.shape)
print("Rentals shape :", df_rentals.shape)
print()
print("Sales columns:")
print(df_sales.columns.tolist())
print()
print("Rentals columns:")
print(df_rentals.columns.tolist())


Sales shape   : (990, 26)
Rentals shape : (999, 27)

Sales columns:
['source_site', 'listing_id', 'listing_url', 'title', 'purchase_price', 'suburb', 'city', 'province', 'property_type', 'bedrooms', 'bathrooms', 'parking_spaces', 'garage', 'floor_area_sqm', 'land_area_sqm', 'levies', 'rates_taxes', 'description', 'listing_date', 'scraped_timestamp', 'pp_transaction_slug', 'pp_province_slug', 'pp_metro_slug', 'pp_city_slug', 'pp_suburb_slug', 'purchase_price_Rands']

Rentals columns:
['source_site', 'listing_id', 'rental_url', 'title', 'monthly_rent', 'suburb', 'city', 'province', 'property_type', 'bedrooms', 'bathrooms', 'parking_spaces', 'garage', 'floor_area_sqm', 'land_area_sqm', 'levies', 'rates_taxes', 'furnished_flag', 'availability_text', 'description', 'listing_date', 'scraped_timestamp', 'pp_transaction_slug', 'pp_province_slug', 'pp_metro_slug', 'pp_city_slug', 'pp_suburb_slug']



## 2. Quick audit before expansion

Before I generate anything, I check the current schema and a few important quality conditions:

- row count
- duplicate IDs
- core price / rent completeness
- whether the target columns are present


In [20]:

def quick_audit(df, id_col, money_col, name):
    print(f"--- {name} ---")
    print("Rows                :", len(df))
    print("Columns             :", len(df.columns))
    print("Duplicate IDs       :", df[id_col].duplicated().sum())
    print(f"Missing {money_col} :", df[money_col].isna().sum())
    print()

quick_audit(df_sales, "listing_id", "purchase_price_Rands", "Sales")
quick_audit(df_rentals, "listing_id", "monthly_rent", "Rentals")


--- Sales ---
Rows                : 990
Columns             : 26
Duplicate IDs       : 0
Missing purchase_price_Rands : 0

--- Rentals ---
Rows                : 999
Columns             : 27
Duplicate IDs       : 0
Missing monthly_rent : 0




## 3. Why this expansion method

A fully generative model such as CTGAN or SDV can be useful, but in this notebook I use a practical and controlled method that is easy to explain and reproduce:

### Method used here
I create synthetic rows by:

1. **sampling real rows with replacement**
2. **preserving the real schema exactly**
3. **adding controlled noise to numeric columns**
4. **keeping categorical values inside observed business ranges**
5. **rebuilding IDs, URLs, titles, and aligned slug columns**
6. **keeping dates inside the observed time window**

### Why this is a good fit here
This works well for my current stage because:

- my real dataset is still relatively small
- I want to preserve suburb, city, and property-type patterns already observed
- I need exact compatibility with downstream cleaning, EDA, and modeling notebooks
- I want a transparent approach that recruiters and reviewers can understand

### What this is not
This is **not** a replacement for more real scraping.  
It is a **training-data expansion strategy**.


In [21]:

def slugify_series(s):
    return (
        s.fillna("")
         .astype(str)
         .str.strip()
         .str.lower()
         .str.replace(r"[^a-z0-9]+", "-", regex=True)
         .str.strip("-")
    )

def parse_listing_date_series(s):
    return pd.to_datetime(s, format="%d %b %Y", errors="coerce")

def perturb_numeric(series, rel_noise=0.05, abs_floor=1.0, integer=False, money=False, clip_low=None, clip_high=None):
    s = series.astype(float).copy()
    base = s.fillna(s.median())
    std = np.nanstd(base)
    noise = np.random.normal(0, max(std * rel_noise, abs_floor), size=len(base))
    out = base + noise

    if clip_low is not None:
        out = np.maximum(out, clip_low)
    if clip_high is not None:
        out = np.minimum(out, clip_high)

    if integer:
        out = np.round(out)

    if money:
        out = np.where(
            out >= 100000,
            np.round(out / 1000) * 1000,
            np.where(out >= 10000, np.round(out / 100) * 100, np.round(out / 50) * 50),
        )

    return pd.Series(out, index=series.index)



## 4. Expansion function

This function expands either dataset to a target row count while keeping:

- exact original columns
- exact original column order
- realistic value ranges
- unique synthetic listing IDs


In [22]:

def expand_fast(df, target_rows, dataset):
    df = df.copy()
    n_add = target_rows - len(df)

    if n_add <= 0:
        return df.iloc[:target_rows].copy()

    sampled = df.sample(n=n_add, replace=True, random_state=42).reset_index(drop=True)
    syn = sampled.copy()

    if dataset == "sales":
        syn_ids = [f"SYNSALE{i:05d}" for i in range(1, n_add + 1)]
        url_col = "listing_url"
        money_col = "purchase_price_Rands"
        trans_slug = "for-sale"
    else:
        syn_ids = [f"SYNRENT{i:05d}" for i in range(1, n_add + 1)]
        url_col = "rental_url"
        money_col = "monthly_rent"
        trans_slug = "to-rent"

    syn["listing_id"] = syn_ids

    # mild category perturbation while staying inside observed values
    for col in [
        "property_type",
        "suburb",
        "city",
        "province",
        "source_site",
        "pp_transaction_slug",
        "pp_province_slug",
        "pp_metro_slug",
        "pp_city_slug",
        "pp_suburb_slug",
    ]:
        if col in syn.columns:
            mask = np.random.rand(n_add) < 0.12
            if mask.sum() > 0:
                syn.loc[mask, col] = df[col].dropna().sample(mask.sum(), replace=True, random_state=7).values

    # re-align slug columns to text columns
    if "province" in syn.columns and "pp_province_slug" in syn.columns:
        syn["pp_province_slug"] = slugify_series(syn["province"])
    if "city" in syn.columns and "pp_city_slug" in syn.columns:
        syn["pp_city_slug"] = slugify_series(syn["city"])
    if "suburb" in syn.columns and "pp_suburb_slug" in syn.columns:
        syn["pp_suburb_slug"] = slugify_series(syn["suburb"])
    if "pp_transaction_slug" in syn.columns:
        syn["pp_transaction_slug"] = trans_slug

    # integer-like property fields
    for c in ["bedrooms", "bathrooms", "parking_spaces", "garage"]:
        if c in syn.columns:
            q99 = df[c].dropna().quantile(0.99) if df[c].dropna().size else None
            syn[c] = perturb_numeric(syn[c], rel_noise=0.04, integer=True, clip_low=0, clip_high=q99)

    # area fields
    for c in ["floor_area_sqm", "land_area_sqm"]:
        if c in syn.columns:
            q99 = df[c].dropna().quantile(0.99) if df[c].dropna().size else None
            syn[c] = perturb_numeric(syn[c], rel_noise=0.06, clip_low=1, clip_high=q99)

    # cost fields
    for c in ["levies", "rates_taxes"]:
        if c in syn.columns:
            if df[c].dropna().empty:
                syn[c] = np.nan
            else:
                q99 = df[c].dropna().quantile(0.99)
                syn[c] = perturb_numeric(syn[c], rel_noise=0.08, clip_low=0, clip_high=q99)

    # main money fields
    if dataset == "sales":
        q99 = df["purchase_price_Rands"].dropna().quantile(0.99)
        syn["purchase_price_Rands"] = perturb_numeric(
            syn["purchase_price_Rands"],
            rel_noise=0.08,
            clip_low=50000,
            clip_high=q99,
            money=True,
        )
        # keep exact relationship seen in the supplied file
        syn["purchase_price"] = syn["purchase_price_Rands"] * 10.0
    else:
        q99 = df["monthly_rent"].dropna().quantile(0.99)
        syn["monthly_rent"] = perturb_numeric(
            syn["monthly_rent"],
            rel_noise=0.08,
            clip_low=1000,
            clip_high=q99,
            money=True,
        )

        if "furnished_flag" in syn.columns:
            p = df["furnished_flag"].dropna().mean() if df["furnished_flag"].dropna().size else np.nan
            syn["furnished_flag"] = np.where(np.random.rand(n_add) < (p if pd.notna(p) else 0), 1.0, np.nan)

        if "availability_text" in syn.columns:
            syn["availability_text"] = syn["availability_text"].astype(object)
            vals = df["availability_text"].dropna()
            mask = np.random.rand(n_add) < 0.45
            syn["availability_text"] = np.nan
            if len(vals):
                syn.loc[mask, "availability_text"] = vals.sample(mask.sum(), replace=True, random_state=11).values

    # business coherence rules
    if set(["bathrooms", "bedrooms"]).issubset(syn.columns):
        syn["bathrooms"] = np.minimum(np.maximum(syn["bathrooms"], 1), syn["bedrooms"].fillna(1) + 2)

    if set(["parking_spaces", "bedrooms"]).issubset(syn.columns):
        syn["parking_spaces"] = np.minimum(np.maximum(syn["parking_spaces"], 1), syn["bedrooms"].fillna(1) + 3)

    if set(["garage", "parking_spaces"]).issubset(syn.columns):
        syn["garage"] = np.minimum(np.maximum(syn["garage"], 1), syn["parking_spaces"].fillna(1))

    if set(["land_area_sqm", "floor_area_sqm"]).issubset(syn.columns):
        mask = syn["land_area_sqm"].notna() & syn["floor_area_sqm"].notna()
        syn.loc[mask, "land_area_sqm"] = np.maximum(
            syn.loc[mask, "land_area_sqm"],
            syn.loc[mask, "floor_area_sqm"] * 0.25,
        )

    # keep dates inside observed history
    parsed_dates = parse_listing_date_series(df["listing_date"]).dropna()
    sampled_dates = parsed_dates.sample(n_add, replace=True, random_state=21).reset_index(drop=True)
    offsets = pd.to_timedelta(np.random.randint(-14, 15, size=n_add), unit="D")
    syn_dates = (sampled_dates + offsets).clip(lower=parsed_dates.min(), upper=parsed_dates.max())
    syn["listing_date"] = syn_dates.dt.strftime("%d %b %Y")

    parsed_scrapes = pd.to_datetime(df["scraped_timestamp"], errors="coerce", utc=True).dropna()
    sampled_scrapes = parsed_scrapes.sample(n_add, replace=True, random_state=22).reset_index(drop=True)
    scrape_offsets = pd.to_timedelta(np.random.randint(-5 * 24 * 60, 5 * 24 * 60, size=n_add), unit="m")
    syn_scrapes = (sampled_scrapes + scrape_offsets).clip(lower=parsed_scrapes.min(), upper=parsed_scrapes.max())
    syn["scraped_timestamp"] = (
        syn_scrapes.dt.strftime("%Y-%m-%dT%H:%M:%S%z")
        .str.replace(r"(\+0000)$", "+00:00", regex=True)
    )

    # rebuild title, description, url
    beds = syn["bedrooms"].fillna(0).astype(int).astype(str)
    syn["title"] = (
        beds
        + " Bedroom "
        + syn["property_type"].fillna("Property").astype(str)
        + " in "
        + syn["suburb"].fillna("").astype(str)
    )

    syn["description"] = (
        df["description"]
        .sample(n_add, replace=True, random_state=23)
        .reset_index(drop=True)
        .astype(str)
        .str.slice(0, 220)
    )

    metro_col = syn["pp_metro_slug"] if "pp_metro_slug" in syn.columns else pd.Series(["johannesburg-metro"] * n_add)

    syn[url_col] = (
        "https://synthetic.privateproperty.co.za/"
        + trans_slug
        + "/"
        + syn["pp_province_slug"].fillna("")
        + "/"
        + metro_col.fillna("johannesburg-metro")
        + "/"
        + syn["pp_city_slug"].fillna("")
        + "/"
        + syn["pp_suburb_slug"].fillna("")
        + "/"
        + syn["listing_id"]
    )

    syn = syn[df.columns]
    out = pd.concat([df, syn], ignore_index=True)
    return out



## 5. Expand both datasets to 10,000 rows each

I keep the original real rows and append synthetic rows until I reach the target size.


In [23]:

TARGET_ROWS = 10_000

expanded_sales = expand_fast(df_sales, TARGET_ROWS, dataset="sales")
expanded_rentals = expand_fast(df_rentals, TARGET_ROWS, dataset="rentals")

print("Expanded sales   :", expanded_sales.shape)
print("Expanded rentals :", expanded_rentals.shape)


C:\Users\ashle\AppData\Local\Temp\ipykernel_8696\2632593404.py:105: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Available Now' 'Available Now' 'Available Now' ...
 'Available 1 April 2026' 'Available Now' 'Available Now']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  syn.loc[mask, "availability_text"] = vals.sample(mask.sum(), replace=True, random_state=11).values


Expanded sales   : (10000, 26)
Expanded rentals : (10000, 27)



## 6. Validate that the schema is exact

This is critical because I want the expanded CSVs to drop directly into the next stages of the pipeline.

I verify:

- exact column names
- exact column order
- unique listing IDs
- no negative sale prices
- no negative rental values


In [24]:

validation_report = {
    "sales_exact_columns": expanded_sales.columns.tolist() == df_sales.columns.tolist(),
    "rentals_exact_columns": expanded_rentals.columns.tolist() == df_rentals.columns.tolist(),
    "sales_unique_listing_id": expanded_sales["listing_id"].is_unique,
    "rentals_unique_listing_id": expanded_rentals["listing_id"].is_unique,
    "sales_positive_price": bool((expanded_sales["purchase_price_Rands"] > 0).all()),
    "rentals_positive_rent": bool((expanded_rentals["monthly_rent"] > 0).all()),
}

validation_report


{'sales_exact_columns': True,
 'rentals_exact_columns': True,
 'sales_unique_listing_id': True,
 'rentals_unique_listing_id': True,
 'sales_positive_price': True,
 'rentals_positive_rent': True}


## 7. Compare original vs expanded distributions

I do not expect the expanded data to be identical, but I do want the broad ranges and medians to stay sensible.

This gives me a quick sanity check on:

- sale price behavior
- rental behavior
- bedroom distributions
- bathroom distributions


In [25]:

summary = pd.DataFrame({
    "sales_original": df_sales[["purchase_price_Rands", "bedrooms", "bathrooms"]].median(),
    "sales_expanded": expanded_sales[["purchase_price_Rands", "bedrooms", "bathrooms"]].median(),
    "rentals_original": df_rentals[["monthly_rent", "bedrooms", "bathrooms"]].median(),
    "rentals_expanded": expanded_rentals[["monthly_rent", "bedrooms", "bathrooms"]].median(),
})

summary


,sales_original,sales_expanded,rentals_original,rentals_expanded
bathrooms,2.0,2.00,2.0,2.0
bedrooms,2.0,2.75,2.0,2.0
monthly_rent,NaN,NaN,12500.0,12400.0
purchase_price_Rands,1250000.0,1295500.00,NaN,NaN



## 8. Export the expanded datasets to CSV

I now save the final expanded datasets with the same schema as the attached cleaned source files.

### Output files
- `expanded_sales_data.csv`
- `expanded_rentals_data.csv`


In [28]:

expanded_sales.to_csv(r"data/processed/expanded_sales_data.csv", index=False)
expanded_rentals.to_csv(r"data/processed/expanded_rentals_data.csv", index=False)

print("Saved:")
print("- expanded_sales_data.csv")
print("- expanded_rentals_data.csv")


Saved:
- expanded_sales_data.csv
- expanded_rentals_data.csv
